In [1]:
import os
import sys
PROJ_ROOT = "/root/autodl-tmp/jump"
sys.path.append(PROJ_ROOT)

import numpy as np
import polars as pl

from datetime import time, datetime, timedelta

from config import DataConfig, get_session_expr
from src.utils import get_raw_data, minus_time
data_cfg = DataConfig()
SYMBOLS = data_cfg.SYMBOLS
EXPR_DAY_SESSIONS = data_cfg.EXPR_DAY_SESSIONS
EXPR_NIGHT_SESSIONS = data_cfg.EXPR_NIGHT_SESSIONS
EXPR_MARKET_OPEN = data_cfg.EXPR_MARKET_OPEN
in_trading_session = EXPR_DAY_SESSIONS | EXPR_NIGHT_SESSIONS

IS_LAST_30MIN = (
    (pl.col("DATA_is_day_session").filter(in_trading_session) != pl.col("DATA_is_day_session").filter(in_trading_session).shift(-30)) |
    (
        (pl.col("DATA_is_day_session").filter(in_trading_session)) & (pl.col("Time").filter(in_trading_session).dt.date() != pl.col("Time").filter(in_trading_session).dt.date().shift(-30))
    )
)

In [2]:
sector = {}
exchange = {}
for s in SYMBOLS:
    if s in ["RB", "HC", "SS", "I", "J", "JM", "SF", "SM"]:
        sector[s] = "heise"    # 黑色系
    elif s in ["AU", "AG"]:
        sector[s] = "guijinshu"    # 贵金属
    elif s in ["CU", "AL", "ZN", "PB", "NI", "SN"]:
        sector[s] = "yousejinshu"    # 有色金属
    elif s in ["SC", "FU", "BU", "PG", "EG", "EB", "L", "V", "PP", "MA", "TA", "SA", "UR", "RU", "NR", "SP", "ZC"]:
        sector[s] = "nengyuanhuagong"    # 能源化工
    elif s in ["A", "B", "M", "Y", "P", "OI", "RM", "C", "CS", "RR", "CF", "CY", "AP", "CJ", "JD", "SR"]:
        sector[s] = "nongfuchanpin"    # 农副产品
    elif s in ["FB", "FG", "SP", "BP", "LW", "GP"]:
        sector[s] = "qinggongjiancai"    # 轻工建材
    else:
        print(s, " not included in sector classification!")

    if s in ["SC", "NR"]:
        exchange[s] = "INE"    # 上期能源
    elif s in ["CU", "AL", "ZN", "PB", "NI", "SN", "AU", "AG", "RB", "HC", "SS", "FU", "BU", "RU", "SP"]:
        exchange[s] = "SHFE"    # 上期所
    elif s in ["I", "J", "JM", "L", "V", "PP", "EG", "EB", "PG", "C", "CS", "A", "B", "M", "Y", "P", "RR", "FB", "JD"]:
        exchange[s] = "DCE"    # 大商所
    elif s in ["CF", "CY", "SR", "AP", "CJ", "OI", "RM", "TA", "MA", "SA", "FG", "ZC", "UR", "SF", "SM"]:
        exchange[s] = "CZCE"    # 郑商所
    else:
        print(s, " not included in exchange classification!")

In [11]:
valid_condition = (
    (pl.col("High") >= pl.col("Low")) & 
    (pl.col("High") >= pl.col("Open")) &
    (pl.col("High") >= pl.col("Close")) &
    (pl.col("Open") >= pl.col("Low")) &
    (pl.col("Close") >= pl.col("Low")) &
    (pl.col("Volume") >=0) & 
    (pl.col("Amount") >=0) & 
    (pl.col("Open Interest") >=0) &
    (pl.col("Open") >= 0) &
    (pl.col("Close") >= 0) &
    (pl.col("High") >= 0) &
    (pl.col("Low") >= 0)
)
symbol_with_null = []
for s in SYMBOLS:
    df = get_raw_data(s, lazy=False)

#--------------------Find rows with null-----------------------------------------------#
    l = len(df.filter(pl.any_horizontal(pl.all().is_null())))
    if l > 0:
        symbol_with_null.append(s)
    print(f"{s} has {l}/{len(df)} rows containing None.")
    
#--------------------Convert time col from string to datetime--------------------------#
    df = df.with_columns([
        pl.col("Time").str.strptime(
            pl.Datetime("us"),
            "%Y-%m-%d %H:%M:%S%.6f",
            strict=False,
        ).dt.truncate("1m"),
    ]).with_columns([
        pl.when(pl.col("Time").dt.hour() >= 20).then(pl.col("Time").dt.date() + pl.duration(days=1))
            .otherwise(pl.col("Time").dt.date()).alias("TradingDay")
    ]).drop_nulls().sort("Time")    # Because the only nulls are the last 20272 lines of FB

#--------------------Print rows violating validity conditions---------------------------#
    bad_df = df.filter(~valid_condition)
    if len(bad_df) > 0:
        print(f"{s} has bad data:")
        display(bad_df)

#--------------------Print rows out of trading hours------------------------------------#
    out_session_df = df.filter(~in_trading_session)
    if len(out_session_df) > 0:
        print(f"{s} has data out of the trading hours:")
        display(out_session_df)

#--------------------Add rows that are in trading hours but missing---------------------#
    dates = df.select(pl.col("TradingDay").alias("date")).unique().to_series().to_list()
    times = df.select(pl.col("Time")).unique().to_series().to_list()
    start_points = [time(9,0), time(10,30), time(13,30), time(21,0)]
    end_points = [
        time(10,14), time(11,29), time(14,59), time(22,59), 
        time(23,29), time(0,59), time(2,29)
    ]
    times_m1 = [dt - timedelta(minutes=1) for dt in times]
    times_p1 = [dt + timedelta(minutes=1) for dt in times]
    extra_times = (
        set(times_p1).intersection(set(times_m1))
    ).union(
        set([datetime.combine(d,t) for d in dates for t in end_points]).intersection(set(times_p1))
    ).union(
        set([datetime.combine(d,t) for d in dates for t in start_points]).intersection(set(times_m1))
    )
    extra_df = pl.DataFrame({"Time": list(extra_times)})

    full_times = pl.concat([df.select("Time"), extra_df]).unique().sort("Time")
    df = full_times.join(df, on="Time", how="left")

#--------------------Add masks to rows--------------------------------------------------#
    df = df.filter(in_trading_session).with_columns([
        pl.lit(s).alias("Symbol"),
        in_trading_session.alias("DATA_is_trading_hour"),
        ((~valid_condition) | (pl.col("Open").is_null())).alias("DATA_is_bad_data"),
        (pl.col("Open").is_null()).alias("DATA_is_missing_data"),
        EXPR_MARKET_OPEN.alias("DATA_is_market_open"),
    ])
    df = df.with_columns([
        pl.when(EXPR_DAY_SESSIONS).then(True)
            .when(EXPR_NIGHT_SESSIONS).then(False)
            .otherwise(None).alias("DATA_is_day_session"),
    ]).with_columns([
        (IS_LAST_30MIN).alias("DATA_is_last_30min"),
    ])

#--------------------Handle bad data---------------------------------------------------#
    bad = pl.col("DATA_is_bad_data") | pl.col("DATA_is_missing_data")
    df_cleaned = df.with_columns([
        pl.when(bad).then(0).otherwise(pl.col("Volume")).alias("Volume"),
        pl.when(bad).then(0).otherwise(pl.col("Amount")).alias("Amount"),
        pl.when(bad & in_trading_session).then(None).otherwise(pl.col("Open")).alias("Open"),
        pl.when(bad & in_trading_session).then(None).otherwise(pl.col("Close")).alias("Close"),
        pl.when(bad & in_trading_session).then(None).otherwise(pl.col("High")).alias("High"),
        pl.when(bad & in_trading_session).then(None).otherwise(pl.col("Low")).alias("Low"),
        pl.when(bad & in_trading_session).then(None).otherwise(pl.col("Open Interest")).alias("Open Interest"),
    ])
    df_cleaned = df_cleaned.with_columns([
        pl.col("Open").forward_fill(),
        pl.col("Close").forward_fill(),
        pl.col("High").forward_fill(),
        pl.col("Low").forward_fill(),
        pl.col("Open Interest").forward_fill(),
        pl.when(pl.col("Time").dt.hour() >= 20).then(pl.col("Time").dt.date() + pl.duration(days=1))
            .otherwise(pl.col("Time").dt.date()).alias("TradingDay"),
    ]).with_columns([
        pl.when(pl.col("TradingDay").dt.weekday() < 5).then(pl.col("TradingDay"))
            .otherwise(pl.col("TradingDay") + pl.duration(days=2)).alias("TradingDay")
    ]).rename({"Open Interest": "OpenInterest"}).drop_nulls()
    df_cleaned = df_cleaned.with_columns([
        pl.when(pl.col("Volume") == 0).then(0).otherwise(pl.col("Volume").log()).alias("LogVolume"),
        pl.when(pl.col("Amount") == 0).then(0).otherwise(pl.col("Amount").log()).alias("LogAmount"),
        pl.col("TradingDay").dt.month().alias("DATA_month"),
        pl.col("TradingDay").dt.weekday().alias("DATA_weekday"),
        pl.lit(sector[s]).alias("DATA_sector"),
        pl.lit(exchange[s]).alias("DATA_exchange")
    ])
    display(df_cleaned.filter(pl.col("DATA_is_missing_data")))

#--------------------Save cleaned data-------------------------------------------------#
    df_cleaned.write_parquet(f"{PROJ_ROOT}/data/cleaned_data/{s}.parquet", compression="zstd")

PP has 0/426905 rows containing None.
PP has data out of the trading hours:


Time,Open,High,Low,Close,Volume,Amount,Open Interest,TradingDay
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date
2014-03-03 08:59:00,1.0,1.0,1.0,1.0,116.0,6.20368e6,21278.0,2014-03-03
2014-03-03 15:00:00,0.994016,0.994016,0.994016,0.994016,0.0,0.0,18328.0,2014-03-03
2014-03-04 15:00:00,1.000935,1.000935,1.000935,1.000935,4.0,214120.0,21784.0,2014-03-04
2014-03-05 08:59:00,1.006082,1.006082,1.006082,1.006082,102.0,5.28309e6,24978.0,2014-03-05
2014-03-05 15:00:00,1.000255,1.000255,1.000255,1.000255,0.0,0.0,23234.0,2014-03-05
…,…,…,…,…,…,…,…,…
2020-04-21 11:30:00,1.320344,1.320344,1.320344,1.320344,25.0,817180.0,433653.0,2020-04-21
2020-04-21 15:00:00,1.334281,1.334281,1.334281,1.334281,4.0,132090.0,428732.0,2020-04-21
2020-04-22 10:15:00,1.304994,1.304994,1.304994,1.304994,12.0,387615.0,452826.0,2020-04-22


Time,Open,High,Low,Close,Volume,Amount,OpenInterest,TradingDay,Symbol,DATA_is_trading_hour,DATA_is_bad_data,DATA_is_missing_data,DATA_is_market_open,DATA_is_day_session,DATA_is_last_30min,LogVolume,LogAmount,DATA_month,DATA_weekday,DATA_sector,DATA_exchange
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date,str,bool,bool,bool,bool,bool,bool,f64,f64,i8,i8,str,str
2014-12-04 13:30:00,0.940747,0.940747,0.940176,0.940176,0.0,0.0,171594.0,2014-12-04,"""PP""",true,true,true,true,true,false,0.0,0.0,12,4,"""nengyuanhuagong""","""DCE"""
2015-07-08 14:21:00,0.96961,0.96961,0.96961,0.96961,0.0,0.0,290752.0,2015-07-08,"""PP""",true,true,true,false,true,false,0.0,0.0,7,3,"""nengyuanhuagong""","""DCE"""
2019-06-28 11:02:00,1.517671,1.517853,1.517307,1.517307,0.0,0.0,592822.0,2019-06-30,"""PP""",true,true,true,false,true,false,0.0,0.0,6,7,"""nengyuanhuagong""","""DCE"""
2019-10-10 21:04:00,1.555659,1.556425,1.555659,1.556233,0.0,0.0,635212.0,2019-10-13,"""PP""",true,true,true,false,false,false,0.0,0.0,10,7,"""nengyuanhuagong""","""DCE"""
2020-02-03 13:43:00,1.372999,1.372999,1.372999,1.372999,0.0,0.0,318581.0,2020-02-03,"""PP""",true,true,true,false,true,false,0.0,0.0,2,1,"""nengyuanhuagong""","""DCE"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2020-03-09 14:09:00,1.360667,1.360667,1.360667,1.360667,0.0,0.0,311209.0,2020-03-09,"""PP""",true,true,true,false,true,false,0.0,0.0,3,1,"""nengyuanhuagong""","""DCE"""
2020-03-09 14:16:00,1.360667,1.360667,1.360667,1.360667,0.0,0.0,311184.0,2020-03-09,"""PP""",true,true,true,false,true,false,0.0,0.0,3,1,"""nengyuanhuagong""","""DCE"""
2020-03-09 14:18:00,1.360667,1.360667,1.360667,1.360667,0.0,0.0,311184.0,2020-03-09,"""PP""",true,true,true,false,true,false,0.0,0.0,3,1,"""nengyuanhuagong""","""DCE"""


Y has 0/613704 rows containing None.
Y has data out of the trading hours:


Time,Open,High,Low,Close,Volume,Amount,Open Interest,TradingDay
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date
2013-09-17 08:59:00,1.0,1.0,1.0,1.0,1730.0,1.227954e8,702164.0,2013-09-17
2013-09-17 15:00:00,1.000564,1.000564,1.000564,1.000564,46.0,3.26776e6,665940.0,2013-09-17
2013-09-18 08:59:00,1.001972,1.001972,1.001972,1.001972,3086.0,2.1947632e8,666602.0,2013-09-18
2013-09-18 15:00:00,0.996055,0.996055,0.996055,0.996055,0.0,0.0,660060.0,2013-09-18
2013-09-23 08:59:00,0.986193,0.986193,0.986193,0.986193,5546.0,3.8822e8,657822.0,2013-09-23
…,…,…,…,…,…,…,…,…
2020-04-23 11:30:00,0.538343,0.538343,0.538343,0.538343,6.0,329360.0,459132.0,2020-04-23
2020-04-23 15:00:00,0.540696,0.540696,0.540696,0.540696,2.0,110280.0,457173.0,2020-04-23
2020-04-24 11:30:00,0.537166,0.537166,0.537166,0.537166,1.0,54780.0,453829.0,2020-04-24


...

SR has 0/592227 rows containing None.
SR has bad data:


Time,Open,High,Low,Close,Volume,Amount,Open Interest,TradingDay
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date
2017-03-20 20:59:00,1.126547,1.126547,1.126547,1.126547,-657160.0,-4.4161e10,449652.0,2017-03-21
2017-03-21 20:59:00,1.113471,1.113471,1.113471,1.113471,-355322.0,-2.3600e10,456088.0,2017-03-22
2018-01-11 20:59:00,1.021835,1.021835,1.021835,1.021835,-222412.0,-1.3013e10,496300.0,2018-01-12
2018-08-13 20:59:00,0.884995,0.884995,0.884995,0.884995,-556432.0,-2.8628e10,375088.0,2018-08-14
2018-09-25 08:59:00,0.843197,0.843197,0.843197,0.843197,-410402.0,-2.0118e10,424688.0,2018-09-25
2018-10-16 20:59:00,0.883275,0.883275,0.883275,0.883275,-728400.0,-3.7403e10,416318.0,2018-10-17
2019-01-02 08:59:00,0.806967,0.806967,0.806967,0.806967,-724350.0,-3.3900e10,482362.0,2019-01-02


SR has data out of the trading hours:


Time,Open,High,Low,Close,Volume,Amount,Open Interest,TradingDay
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date
2013-09-30 08:59:00,1.0,1.0,0.999803,0.999803,1782.0,9.024048e7,559534.0,2013-09-30
2013-10-09 08:59:00,1.018164,1.018164,1.018164,1.018164,770.0,3.97089e7,511060.0,2013-10-09
2013-10-10 08:59:00,1.014215,1.014215,1.014215,1.014215,260.0,1.33562e7,522186.0,2013-10-10
2013-10-10 15:00:00,1.032577,1.032774,1.032577,1.032774,120.0,6.2772e6,597680.0,2013-10-10
2013-10-11 08:59:00,1.034353,1.034353,1.034353,1.034353,2412.0,1.2636468e8,596520.0,2013-10-11
…,…,…,…,…,…,…,…,…
2021-01-26 20:59:00,0.942481,0.942481,0.942481,0.942481,559.0,2.944812e6,457092.0,2021-01-27
2021-01-27 20:59:00,0.948206,0.948206,0.948206,0.948206,841.0,4.4573e6,463783.0,2021-01-28
2021-01-28 20:59:00,0.935682,0.935682,0.935682,0.935682,1360.0,7.1128e6,462076.0,2021-01-29


Time,Open,High,Low,Close,Volume,Amount,OpenInterest,TradingDay,Symbol,DATA_is_trading_hour,DATA_is_bad_data,DATA_is_missing_data,DATA_is_market_open,DATA_is_day_session,DATA_is_last_30min,LogVolume,LogAmount,DATA_month,DATA_weekday,DATA_sector,DATA_exchange
datetime[μs],f64,f64,f64,f64,f64,f64,f64,date,str,bool,bool,bool,bool,bool,bool,f64,f64,i8,i8,str,str
2014-10-16 10:04:00,0.933046,0.933248,0.93244,0.93244,0.0,0.0,787506.0,2014-10-16,"""SR""",true,true,true,false,true,false,0.0,0.0,10,4,"""nongfuchanpin""","""CZCE"""
2015-07-15 21:35:00,1.015578,1.015578,1.01501,1.015199,0.0,0.0,376288.0,2015-07-16,"""SR""",true,true,true,false,false,false,0.0,0.0,7,4,"""nongfuchanpin""","""CZCE"""
2016-01-12 22:59:00,1.007785,1.008153,1.007785,1.007969,0.0,0.0,950896.0,2016-01-13,"""SR""",true,true,true,false,false,true,0.0,0.0,1,3,"""nongfuchanpin""","""CZCE"""
2018-07-19 21:03:00,0.869374,0.870082,0.869198,0.869198,0.0,0.0,399080.0,2018-07-22,"""SR""",true,true,true,false,false,false,0.0,0.0,7,7,"""nongfuchanpin""","""CZCE"""
2019-05-23 10:34:00,0.852369,0.853235,0.852369,0.853235,0.0,0.0,603670.0,2019-05-23,"""SR""",true,true,true,false,true,false,0.0,0.0,5,4,"""nongfuchanpin""","""CZCE"""
2019-06-28 11:02:00,0.869704,0.869704,0.869184,0.86953,0.0,0.0,509556.0,2019-06-30,"""SR""",true,true,true,false,true,false,0.0,0.0,6,7,"""nongfuchanpin""","""CZCE"""
2020-02-04 13:46:00,0.993841,0.993841,0.993841,0.993841,0.0,0.0,263975.0,2020-02-04,"""SR""",true,true,true,false,true,false,0.0,0.0,2,2,"""nongfuchanpin""","""CZCE"""


In [13]:
s = "I"
d = pl.datetime(2014,12,27)
with pl.Config(tbl_rows=1000):
    display(
        get_raw_data(s).with_columns([
            pl.col("Time").str.strptime(
                pl.Datetime("us"),
                "%Y-%m-%d %H:%M:%S%.6f",
                strict=False
            ).dt.truncate("1m"),
        ]).filter(pl.col("Time").dt.date() == d)
    )

Time,Open,High,Low,Close,Volume,Amount,Open Interest
datetime[μs],f64,f64,f64,f64,f64,f64,f64
2014-12-27 00:00:00,0.546384,0.547531,0.546384,0.546384,60.0,2.8564e6,697312.0
2014-12-27 00:01:00,0.546384,0.547531,0.546384,0.546384,10.0,476200.0,697316.0
2014-12-27 00:02:00,0.546384,0.546384,0.546384,0.546384,18.0,856800.0,697330.0
2014-12-27 00:03:00,0.546384,0.546384,0.546384,0.546384,10.0,476000.0,697340.0
2014-12-27 00:04:00,0.546384,0.546384,0.546384,0.546384,4.0,190400.0,697344.0
2014-12-27 00:05:00,0.546384,0.546384,0.546384,0.546384,270.0,1.2852e7,697370.0
2014-12-27 00:06:00,0.546384,0.546384,0.546384,0.546384,16.0,761600.0,697376.0
2014-12-27 00:07:00,0.546384,0.546384,0.546384,0.546384,70.0,3.332e6,697382.0
2014-12-27 00:08:00,0.546384,0.546384,0.546384,0.546384,24.0,1.1424e6,697360.0


In [14]:
symbol_with_null

['FB']

In [16]:
def add_return(s: str, w: int, y_old=None):
    df = pl.read_parquet(
        f"{PROJ_ROOT}/data/cleaned_data/{s}.parquet"
    ).select([
        "Time", "Symbol", "Close", "DATA_is_trading_hour", 
        "DATA_is_day_session", "DATA_is_bad_data", "DATA_is_last_30min"
    ])
    df_w = df.select(["Time", "Close"]).with_columns([
        (pl.col("Time") - pl.duration(minutes=w)).alias("Time")
    ]).rename({"Close": f"Close_t+{w}"})
    df_w15 = df.select(["Time", "Close"]).with_columns([
        (pl.col("Time") - pl.duration(minutes=15+w)).alias("Time")
    ]).rename({"Close": f"Close_t+{15+w}"})
    df_w120 = df.select(["Time", "Close"]).with_columns([
        (pl.col("Time") - pl.duration(minutes=120+w)).alias("Time")
    ]).rename({"Close": f"Close_t+{120+w}"})
    df = df.join(df_w, on="Time", how="left").join(df_w15, on="Time", how="left").join(df_w120, on="Time", how="left")
    y = df.with_columns([
        pl.when(
            pl.col("DATA_is_last_30min")
        ).then(None).when(
            get_session_expr(minus_time(time(10,14),w-1), time(10,14))
        ).then(
            (pl.col(f"Close_t+{15+w}") / pl.col("Close") - 1)
        ).when(
            get_session_expr(minus_time(time(11,29),w-1), time(11,29))
        ).then(
            (pl.col(f"Close_t+{120+w}") / pl.col("Close") - 1)
        ).otherwise(
            (pl.col(f"Close_t+{w}") / pl.col("Close") - 1)
        ).alias(f"ret_{w}min")
    ]).select(["Time", "Symbol", f"ret_{w}min"])
    if y_old is None:
        return y
    else:
        return y_old.join(y, on=["Time", "Symbol"], how="left")

In [18]:
from tqdm.auto import tqdm
for s in tqdm(SYMBOLS):
    for w in range(1,31):
        if w == 1:
            y_old = add_return(s,w)
        else:
            y_old = add_return(s,w,y_old)
    y_old.write_parquet(f"{PROJ_ROOT}/data/labels/{s}.parquet", compression="zstd")

  0%|          | 0/51 [00:00<?, ?it/s]